This script prepares prompt-chatbot exchanges for semantic distance calculations by removing emojis, artefacts, and short strings

The raw chatlog files are unavailable due to privacy concerns. However, the script used to compute the measures (this script) has been made available for purview. 

In [ ]:
import pandas as pd
import numpy as np
from itables import show
import re
import os
import emoji

root_dir = Path('/rootdir')

In [ ]:
tgaia_data = root_dir / "02_processed_data" / "output" / "selected_tgaia.xlsx"
tgaia_full = pd.read_excel(tgaia_data)
tgaia_output = tgaia_full # copy of the original dataframe because i'm anxious about overwriting the original dataframe

# Functions

In [ ]:
# remove emojis
def remove_emojis(text):
    return emoji.replace_emoji(text, replace='')

In [ ]:
# mask to remove short strings
def filter_short_text(df, column_name):
    mask = df[column_name].astype(str).str.split().str.len() >= 2
    return df[mask].copy()

In [ ]:
# cleaner for excel artefacts and weird teacherGAIA formatting quirks

def helper_clean(text):
    # why does python do this
    text = text.replace('_x000D_', ' ')

    # (?:^|(?<=\n))   -> Start of string OR Lookbehind for newline
    # \s*             -> Optional whitespace before the marker
    # (?:             -> Start non-capturing group for marker types
    #   [-*]          -> Match a bullet (hyphen or asterisk)
    #   |             -> OR
    #   \d+[.)]       -> Match digits followed by a dot OR a closing parenthesis
    # )               -> End marker group
    # \s+             -> Match the whitespace after the marker
    
    pattern = r'(?:^|(?<=\n))\s*(?:[-*]|\d+[.)])\s+'
    text = re.sub(pattern, '', text)
    
    # no bold / headers
    text = text.replace('**', '')

    # no new lines
    text = text.replace('\n', ' ')

    # no extra  w h i t e s p a c e 
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
# split data by dataset level and write them to the specified folder

def split_and_save_dataframe(dataframe, column_name, output_directory='/Users/brainmemorylab/NUS Dropbox/BrainMemLab_TeamFolder/Projects/Tidy_TeacherGAIA/03_methods/semantic_distance/inputs'):
    if column_name not in dataframe.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame.")

    os.makedirs(output_directory, exist_ok=True)

    for group_name, group_df in dataframe.groupby(column_name):
        filename = f"preprocessed_for_embeddings_{group_name}.pkl"
        file_path = os.path.join(output_directory, filename)
        
        group_df.to_pickle(file_path)
        print(f"Saved {len(group_df)} rows to {file_path}")

# Actual Output

In [ ]:
# remove emojis
tgaia_output['prompt'] = tgaia_output['prompt'].apply(remove_emojis)
tgaia_output['response'] = tgaia_output['response'].apply(remove_emojis)

In [ ]:
# clean out excel artefacts and teacherGAIA formatting from 'prompt'
tgaia_output['prompt'] = tgaia_output['prompt'].apply(helper_clean)
show(tgaia_output['prompt'])

In [ ]:
# clean out excel artefacts and teacherGAIA formatting from 'response'
tgaia_output['response'] = tgaia_output['response'].apply(helper_clean)
show(tgaia_output['response'])

In [ ]:
# clean out short strings
tgaia_output = filter_short_text(tgaia_output, 'prompt')
show(tgaia_output['prompt'])

# Outputs

## Split by dataset

In [ ]:
check_dataset_levels = tgaia_output['dataset'].unique()
print(check_dataset_levels)

In [ ]:
split_and_save_dataframe(tgaia_output, 'dataset')

## A full one

In [ ]:
out_dir = root_dir / "03_methods" / "semantic_distance" / "inputs"
tgaia_output.to_pickle(out_dir / "preprocessed_for_embeddings_full.pkl")
tgaia_output.to_excel(out_dir / "preprocessed_for_embeddings_full.xlsx", index=False)